# Trend Tracker — Notebook 02: Semantic Enrichment

Builds two injection maps from the preprocessed vocabulary, then writes an
enriched parquet that `03_insights_generation` can load in place of `05_filtered.parquet`.

**Run order:** `01_token_preprocess` → `02_semantic_enrichment` → `03_insights_generation`

| Pass | What | Key outputs |
|------|------|-------------|
| A | Subject clustering (SentenceTransformer + agglomerative + LLM gate) | `enrichment/semantic_map.csv` |
| B | Framing taxonomy classification (LLM batch) | `enrichment/framing_map.csv` |
| C | Token injection | `prepared/06_enriched.parquet` |

**Recommended spot checks:**
- After Pass A: review `semantic_map.csv` — term → subject category mappings.
- Before Pass B: confirm `framing_taxonomy.json` is the intended taxonomy version.
- After Pass B: review `framing_map.csv` and any coverage deviation warnings.
- After Pass C: check injection stats and decide whether to use the enriched or filtered parquet downstream.

---
## Setup and Configuration

In [1]:
import os
import sys
import json
import yaml
import hashlib
import pickle
import re
from collections import Counter
from datetime import datetime
from pathlib import Path

import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.cluster import AgglomerativeClustering
import httpx
import urllib3

ROOT = Path(".")
sys.path.insert(0, str(ROOT))

import importlib.util as _ilu, sys as _sys
_u = next(Path(".").glob("utils*.py"))
if _u.stem != "utils":
    _s = _ilu.spec_from_file_location("utils", _u); _m = _ilu.module_from_spec(_s); _s.loader.exec_module(_m); _sys.modules["utils"] = _m
    
from utils import (
    load_cfg,
    flat_freq,
    build_output_path,
    get_llm_client,
    ensure_warning_file,
    append_warning,
    start_stage_manifest,
    finalize_stage_manifest,
    artifact_meta,
    gate_cluster,
    classify_batch,
    inject_tokens,
    resolve_params_path,
)

CFG_PATH = resolve_params_path()
CFG = load_cfg(CFG_PATH)

try:
    MANIFEST_CFG_PATH = str(CFG_PATH.relative_to(ROOT))
except ValueError:
    MANIFEST_CFG_PATH = str(CFG_PATH)

# Convenience wrapper: NB02 outputs are not run-scoped.
def OUT(subdir, fname):
    return build_output_path(subdir, fname)

# ── Proxy environment: SSL verification bypass ───────────────────────────────
# SentenceTransformer downloads models via the HuggingFace Hub (httpx +
# requests). The DonorsChoose proxy requires SSL verification to be disabled
# for outbound model downloads. This patch is idempotent and only runs once
# per kernel session.
os.environ["CURL_CA_BUNDLE"]    = ""
os.environ["REQUESTS_CA_BUNDLE"] = ""

if not getattr(httpx.Client, "_dc_verify_patched", False):
    _orig_init = httpx.Client.__init__
    def _patched_init(self, *args, **kwargs):
        kwargs["verify"] = False
        return _orig_init(self, *args, **kwargs)
    httpx.Client.__init__ = _patched_init
    httpx.Client._dc_verify_patched = True

urllib3.disable_warnings()

# ── Warnings, manifests, LLM client ──────────────────────────────────────────
WARNINGS_PATH = OUT("enrichment", "warnings_02.jsonl")
ensure_warning_file(WARNINGS_PATH)

STAGE_MANIFEST = start_stage_manifest(
    stage_name="02_semantic_enrichment",
    notebook_file="02_semantic_enrichment_v1.ipynb",
    config_path=MANIFEST_CFG_PATH,
    run_id=None,
    group_by_field=None,
    filter_fields_key=None,
)

client         = get_llm_client()
MODEL_GATE     = CFG["models"]["gate"]
MODEL_CLASSIFY = CFG["models"]["classify"]

# ── Load corpus and build vocabulary artifacts ────────────────────────────────
# We build two vocabulary frames:
#
#   unigram_vocab_df:
#       Unigram-only vocabulary. Used for Pass A rare-term semantic clustering.
#
#   taxonomy_vocab_artifact_df:
#       Validation/enrichment vocabulary containing unigrams plus TF-IDF-style
#       n-gram candidates. Used as a reservoir for taxonomy phrase anchors.
#
# Important:
#   Pass B should NOT classify all taxonomy_vocab_artifact_df rows. It will later
#   classify unigrams plus only taxonomy phrase anchors that exist in this artifact.

import numpy as np
from sklearn.feature_extraction.text import CountVectorizer

def normalize_vocab_term(x):
    if pd.isna(x):
        return ""
    s = str(x).strip().lower()
    s = s.replace("_", " ")
    s = re.sub(r"\s+", " ", s)
    return s.strip(" \t\r\n\"'`.,;:()[]{}")

def term_type(term):
    n = len(str(term).split())
    if n <= 1:
        return "unigram"
    if n == 2:
        return "bigram"
    return "trigram_or_longer"

def parse_token_list_for_vocab(x):
    if isinstance(x, (list, tuple, set, np.ndarray)):
        return [normalize_vocab_term(t) for t in list(x) if normalize_vocab_term(t)]
    if x is None or pd.isna(x):
        return []
    s = str(x).strip()
    if not s:
        return []
    if "," in s:
        return [normalize_vocab_term(t) for t in s.split(",") if normalize_vocab_term(t)]
    return [normalize_vocab_term(t) for t in s.split() if normalize_vocab_term(t)]

df = pd.read_parquet(ROOT / "OUTPUTS/prepared/05_filtered.parquet")

# ── Unigram vocabulary, preserved for Pass A and backward compatibility ──────
doc_freq = (
    pd.read_csv(ROOT / "OUTPUTS/vocab/unigram_docfreq.csv", index_col=0)
    .squeeze()
    .rename("doc_freq")
)
freq = flat_freq(df).rename("corpus_freq")

unigram_vocab_df = pd.DataFrame({"corpus_freq": freq, "doc_freq": doc_freq}).dropna()
unigram_vocab_df.index = unigram_vocab_df.index.map(normalize_vocab_term)
unigram_vocab_df = unigram_vocab_df[unigram_vocab_df.index != ""]
unigram_vocab_df = unigram_vocab_df[~unigram_vocab_df.index.duplicated(keep="first")]
unigram_vocab_df["docs_pct"] = unigram_vocab_df["doc_freq"] / len(df)
unigram_vocab_df["term_type"] = "unigram"
unigram_vocab_df["source"] = "filtered_tokens"

# Keep vocab_df as unigrams for Pass A. A later cell will replace vocab_df with
# the Pass B classification vocabulary after the taxonomy is loaded.
vocab_df = unigram_vocab_df.copy()

# ── N-gram vocabulary artifact for taxonomy phrase anchors ───────────────────
tokens_by_doc = df["tokens"].apply(parse_token_list_for_vocab)
docs_for_ngrams = tokens_by_doc.apply(lambda ts: " ".join(ts)).tolist()

tfidf_cfg = CFG.get("tfidf", {})
ngram_cfg = CFG.get("ngrams", {})

ngram_min_df = int(tfidf_cfg.get("min_df", CFG["sql"]["min_doc_count"]))
ngram_max_df = float(tfidf_cfg.get("max_df", CFG["sql"]["max_doc_fraction"]))
ngram_max_n  = int(ngram_cfg.get("max_n", 2))
ngram_max_n  = max(1, min(ngram_max_n, 3))

ngram_df = pd.DataFrame(columns=[
    "term", "doc_freq", "corpus_freq", "docs_pct", "term_type", "source"
])

if ngram_max_n >= 2:
    vec = CountVectorizer(
        lowercase=False,
        token_pattern=r"(?u)\b\S+\b",
        ngram_range=(1, ngram_max_n),
        min_df=ngram_min_df,
        max_df=ngram_max_df,
    )

    X = vec.fit_transform(docs_for_ngrams)
    terms = vec.get_feature_names_out()

    ngram_df = pd.DataFrame({
        "term": [normalize_vocab_term(t) for t in terms],
        "doc_freq": (X > 0).sum(axis=0).A1.astype(int),
        "corpus_freq": X.sum(axis=0).A1.astype(int),
        "source": "tfidf_ngram_candidate",
    })
    ngram_df = ngram_df[ngram_df["term"] != ""].copy()
    ngram_df["docs_pct"] = ngram_df["doc_freq"] / len(df)
    ngram_df["term_type"] = ngram_df["term"].apply(term_type)

# Combine unigram rows and n-gram rows. Prefer the direct unigram stats when a
# unigram also appears in the CountVectorizer output.
unigram_artifact_df = (
    unigram_vocab_df
    .reset_index()
    .rename(columns={"index": "term"})
    [["term", "doc_freq", "corpus_freq", "docs_pct", "term_type", "source"]]
)

taxonomy_vocab_artifact_df = pd.concat(
    [unigram_artifact_df, ngram_df],
    ignore_index=True
)

taxonomy_vocab_artifact_df["term"] = taxonomy_vocab_artifact_df["term"].map(normalize_vocab_term)
taxonomy_vocab_artifact_df = taxonomy_vocab_artifact_df[taxonomy_vocab_artifact_df["term"] != ""].copy()

taxonomy_vocab_artifact_df["source_rank"] = taxonomy_vocab_artifact_df["source"].map({
    "filtered_tokens": 0,
    "tfidf_ngram_candidate": 1,
}).fillna(9)

taxonomy_vocab_artifact_df = (
    taxonomy_vocab_artifact_df
    .sort_values(["term", "source_rank", "doc_freq"], ascending=[True, True, False])
    .drop_duplicates(subset=["term"], keep="first")
    .drop(columns=["source_rank"])
    .sort_values(["term_type", "doc_freq", "term"], ascending=[True, False, True])
    .reset_index(drop=True)
)

taxonomy_vocab_artifact_path = ROOT / "OUTPUTS/vocab/taxonomy_vocab_artifact.csv"
taxonomy_vocab_artifact_df.to_csv(taxonomy_vocab_artifact_path, index=False)

print(f"Corpus                         : {len(df):,} projects")
print(f"Pass A unigram vocab            : {len(unigram_vocab_df):,} terms")
print(f"Taxonomy vocab artifact         : {len(taxonomy_vocab_artifact_df):,} terms")
print(f"  Unigrams                      : {(taxonomy_vocab_artifact_df['term_type'] == 'unigram').sum():,}")
print(f"  Bigrams                       : {(taxonomy_vocab_artifact_df['term_type'] == 'bigram').sum():,}")
print(f"  Trigrams/longer               : {(taxonomy_vocab_artifact_df['term_type'] == 'trigram_or_longer').sum():,}")
print(f"Taxonomy vocab artifact saved → {taxonomy_vocab_artifact_path}")
print(f"Warnings                        : {WARNINGS_PATH}")

Corpus                         : 2,244,916 projects
Pass A unigram vocab            : 7,553 terms
Taxonomy vocab artifact         : 3,392,511 terms
  Unigrams                      : 7,553
  Bigrams                       : 1,610,995
  Trigrams/longer               : 1,773,963
Taxonomy vocab artifact saved → OUTPUTS/vocab/taxonomy_vocab_artifact.csv
Warnings                        : /Users/matt.fritz/Desktop/Research Insights/Essay Prototype/OUTPUTS/enrichment/warnings_02.jsonl


In [2]:
# ── Diagnostic: is Pass B classifying unigrams only? ─────────────────────────
# Run this after vocab_df exists, and again after framing_map exists.

import json
import re
from pathlib import Path
import pandas as pd

def normalize_term(x):
    return re.sub(r"\s+", " ", str(x).strip().lower())

def classify_term_type(term):
    term = normalize_term(term)
    n = len(term.split())
    if n <= 1:
        return "unigram"
    if n == 2:
        return "bigram"
    return "trigram_or_longer"

def extract_vocab_terms(vocab_df):
    """
    Robustly extract the vocabulary terms that Pass B can classify.
    Handles either:
      - vocab_df index = terms
      - vocab_df['term']
      - vocab_df['vocab_term']
    """
    if "term" in vocab_df.columns:
        terms = vocab_df["term"]
    elif "vocab_term" in vocab_df.columns:
        terms = vocab_df["vocab_term"]
    else:
        terms = pd.Series(vocab_df.index)

    return (
        terms
        .dropna()
        .astype(str)
        .map(normalize_term)
        .loc[lambda s: s.ne("")]
        .drop_duplicates()
    )

# 1. What terms are available to Pass B?
vocab_terms = extract_vocab_terms(vocab_df)

vocab_term_type_summary = (
    vocab_terms
    .map(classify_term_type)
    .value_counts()
    .rename_axis("term_type")
    .reset_index(name="count")
)

print("Vocabulary terms available to Pass B:")
display(vocab_term_type_summary)

print("\nExample non-unigram vocabulary terms available to Pass B:")
display(vocab_terms[vocab_terms.str.contains(r"\s", regex=True)].head(50).to_frame("term"))

# 2. What terms did Pass B actually classify into taxonomy categories?
if "framing_map" in globals():
    if "term" not in framing_map.columns:
        print("\nframing_map exists but has no 'term' column. Columns:", list(framing_map.columns))
    else:
        mapped_terms = (
            framing_map["term"]
            .dropna()
            .astype(str)
            .map(normalize_term)
            .loc[lambda s: s.ne("")]
        )

        framing_map_type_summary = (
            mapped_terms
            .map(classify_term_type)
            .value_counts()
            .rename_axis("term_type")
            .reset_index(name="mapped_term_count")
        )

        print("\nTerms actually mapped by Pass B:")
        display(framing_map_type_summary)

        print("\nExample non-unigram terms actually mapped by Pass B:")
        display(mapped_terms[mapped_terms.str.contains(r"\s", regex=True)].head(50).to_frame("mapped_term"))
else:
    print("\nframing_map does not exist yet. Run the Pass B classification cell, then rerun this diagnostic.")

# 3. Are phrase examples/counter-examples from the taxonomy even present in the Pass B vocabulary?
taxonomy_path = OUT("enrichment", "framing_taxonomy.json") if "OUT" in globals() else Path("OUTPUTS/enrichment/framing_taxonomy.json")

if Path(taxonomy_path).exists():
    taxonomy = json.loads(Path(taxonomy_path).read_text())

    phrase_rows = []
    vocab_set = set(vocab_terms)

    for entry in taxonomy:
        category = entry.get("category", "")
        for role in ["examples", "counter_examples"]:
            for term in entry.get(role, []):
                t = normalize_term(term)
                if " " in t:
                    phrase_rows.append({
                        "category": category,
                        "role": role,
                        "term": t,
                        "term_type": classify_term_type(t),
                        "present_in_pass_b_vocab": t in vocab_set,
                    })

    phrase_check = pd.DataFrame(phrase_rows)

    print("\nTaxonomy phrase anchors present in Pass B vocabulary:")
    if phrase_check.empty:
        print("No phrase examples/counter-examples found in taxonomy.")
    else:
        display(
            phrase_check
            .groupby(["role", "term_type", "present_in_pass_b_vocab"])
            .size()
            .reset_index(name="count")
            .sort_values(["role", "term_type", "present_in_pass_b_vocab"])
        )

        print("\nPhrase anchors missing from Pass B vocabulary:")
        display(
            phrase_check[~phrase_check["present_in_pass_b_vocab"]]
            .sort_values(["category", "role", "term"])
            .head(100)
        )
else:
    print(f"\nCould not find taxonomy file at: {taxonomy_path}")

Vocabulary terms available to Pass B:


,term_type,count
0,unigram,7553



Example non-unigram vocabulary terms available to Pass B:


,term



framing_map does not exist yet. Run the Pass B classification cell, then rerun this diagnostic.

Taxonomy phrase anchors present in Pass B vocabulary:


,role,term_type,present_in_pass_b_vocab,count
0,counter_examples,bigram,False,392
1,examples,bigram,False,766



Phrase anchors missing from Pass B vocabulary:


,category,role,term,term_type,present_in_pass_b_vocab
11,context_alternative_school_justice_adjacent_co...,counter_examples,basketball court,bigram,False
10,context_alternative_school_justice_adjacent_co...,counter_examples,mock trial,bigram,False
14,context_alternative_school_justice_adjacent_co...,counter_examples,racial justice,bigram,False
13,context_alternative_school_justice_adjacent_co...,counter_examples,social justice,bigram,False
12,context_alternative_school_justice_adjacent_co...,counter_examples,tennis court,bigram,False
...,...,...,...,...,...
91,framing_applied_production_real_world_execution,examples,final product,bigram,False
92,framing_applied_production_real_world_execution,examples,multi step,bigram,False
90,framing_applied_production_real_world_execution,examples,project completion,bigram,False
93,framing_applied_production_real_world_execution,examples,step world,bigram,False


---
## Pass A — Subject Clustering

Embeds rare-vocabulary terms with SentenceTransformer, clusters them with
agglomerative cosine-distance clustering, then runs each cluster through an
LLM coherence gate.

Gate decisions:
- **inject** — terms share a coherent educational domain concept; a `__cat_*__`
  token is added to any project containing a member term.
- **split** — 2–3 coherent sub-groups exist; each sub-group is treated as a
  separate inject.
- **discard** — generic language, morphological noise, or no meaningful shared
  concept.

Embeddings are cached by input hash so re-runs skip the encode step.

In [3]:
RARE_DOC_FREQ_CEILING  = CFG["enrichment"]["rare_doc_freq_ceiling"]
AGG_DISTANCE_THRESHOLD = CFG["enrichment"]["agg_distance_threshold"]
MIN_CLUSTER_SIZE       = CFG["enrichment"]["min_cluster_size"]
EMBEDDING_MODEL        = CFG["enrichment"]["embedding_model"]
CACHE_SCHEMA_VERSION   = CFG["enrichment"]["cache_schema_version"]

# ── Select rare terms to cluster ──────────────────────────────────────────────
rare_terms = vocab_df[vocab_df["doc_freq"] < RARE_DOC_FREQ_CEILING].index.tolist()
print(f"Rare terms to cluster: {len(rare_terms):,}")

# ── Build a cache key from inputs ─────────────────────────────────────────────
# Any change to the term list, model, thresholds, or schema version produces
# a new key and forces a fresh encode + cluster run.
cache_components = "|".join([
    hashlib.md5(",".join(sorted(rare_terms)).encode()).hexdigest(),
    EMBEDDING_MODEL,
    str(AGG_DISTANCE_THRESHOLD),
    str(MIN_CLUSTER_SIZE),
    CACHE_SCHEMA_VERSION,
])
cache_key  = hashlib.md5(cache_components.encode()).hexdigest()
CACHE_DIR  = ROOT / ".cache"
CACHE_DIR.mkdir(exist_ok=True)
cache_path = CACHE_DIR / f"embeddings_clusters_{cache_key[:12]}.pkl"

# ── Guard: skip Pass A when too few rare terms exist ────────────────────────
# AgglomerativeClustering requires at least 2 samples. When rare_terms is empty
# or contains only 1 term — possible in narrow or heavily filtered runs — the
# clustering call would raise a ValueError. We short-circuit here and produce
# empty-but-valid outputs so downstream cells continue without error.
if len(rare_terms) < 2:
    append_warning(
        WARNINGS_PATH, "02_semantic_enrichment", "PASS_A_SKIPPED",
        f"Pass A skipped: only {len(rare_terms)} rare term(s) below ceiling "
        f"(minimum 2 required for clustering).",
        context={"rare_term_count": len(rare_terms), "ceiling": RARE_DOC_FREQ_CEILING},
    )
    print(
        f"Pass A skipped — only {len(rare_terms)} rare term(s) found below "
        f"doc_freq ceiling {RARE_DOC_FREQ_CEILING}. "
        "Minimum 2 required for clustering."
    )
    # Produce empty-but-valid outputs so the gate and semantic_map cells continue.
    clusters = pd.DataFrame(columns=["cluster_id", "terms"])

# ── Embed and cluster (or load from cache) ────────────────────────────────────
elif cache_path.exists():
    print(f"Cache hit ({cache_key[:8]}) — loading embeddings and cluster labels...")
    with open(cache_path, "rb") as f:
        embeddings, labels = pickle.load(f)
else:
    print(f"Cache miss — embedding {len(rare_terms):,} terms with {EMBEDDING_MODEL}...")
    embedder   = SentenceTransformer(EMBEDDING_MODEL)
    embeddings = embedder.encode(
        rare_terms, show_progress_bar=True, normalize_embeddings=True
    )
    agg = AgglomerativeClustering(
        n_clusters=None,
        distance_threshold=AGG_DISTANCE_THRESHOLD,
        metric="cosine",
        linkage="average",
    )
    labels = agg.fit_predict(embeddings)
    with open(cache_path, "wb") as f:
        pickle.dump((embeddings, labels), f)
    print(f"Written to cache: {cache_path.name}")

# ── Filter to valid clusters and inspect ─────────────────────────────────────
sizes     = Counter(labels)
valid_ids = {cid for cid, n in sizes.items() if n >= MIN_CLUSTER_SIZE}
cluster_df = pd.DataFrame({"term": rare_terms, "cluster_id": labels})
cluster_df = cluster_df[cluster_df["cluster_id"].isin(valid_ids)].copy()
clusters   = (
    cluster_df.groupby("cluster_id")["term"].apply(list).reset_index(name="terms")
)

print(f"\nTerms in valid clusters : {len(cluster_df):,} of {len(rare_terms):,}")
print(f"Valid clusters ({MIN_CLUSTER_SIZE}+ members): {len(clusters)}")
print("\nSample clusters:")
for _, row in clusters.sample(min(8, len(clusters)), random_state=42).iterrows():
    print(f"  C{row.cluster_id:03d}: {row.terms[:12]}")

Rare terms to cluster: 2,431
Cache hit (2c551767) — loading embeddings and cluster labels...

Terms in valid clusters : 94 of 2,431
Valid clusters (3+ members): 30

Sample clusters:
  C405: ['gumball', 'gummies', 'gummy']
  C096: ['candidate', 'nominate', 'nominee']
  C198: ['apparel', 'garment', 'textile']
  C126: ['brazil', 'england', 'france', 'germany', 'italy', 'spain']
  C055: ['aunt', 'cousin', 'uncle']
  C057: ['el', 'elapse', 'elar', 'elate']
  C442: ['battleship', 'naval', 'navy']
  C204: ['defend', 'defense', 'defensive']


In [4]:
# ── LLM coherence gate ────────────────────────────────────────────────────────
# One API call per cluster. gate_cluster() is defined in utils.py and handles
# retry / error logic. Prompts are defined here to keep domain language visible.

GATE_SYSTEM = (
    "You are evaluating vocabulary clusters extracted from DonorsChoose K-12 teacher "
    "project essays for a semantic enrichment pipeline.\n\n"
    "Your decisions control which clusters become injected enrichment tokens "
    "(for example: __cat_marine_biology__). These injected tokens are appended to "
    "project token lists and then used downstream as TF-IDF and NMF signals in topic "
    "modeling.\n\n"
    "Precision matters more than recall. A false inject approves a noisy, generic, or "
    "mixed cluster and pollutes downstream topics with meaningless signal. A false "
    "discard is acceptable. Under-injection is cheaper than over-injection.\n\n"
    "Use short snake_case labels when labeling coherent concepts.\n"
    "Return ONLY valid JSON. No markdown, no code fences, no extra keys."
)

GATE_PROMPT = (
    "Cluster {cid} terms: {terms}\n\n"
    "Decide whether these terms form a coherent, specific educational domain concept "
    "worth surfacing as an injected topic-modeling signal.\n\n"
    "Actions:\n"
    "inject:\n"
    "- Use only when a clear majority of terms cohere around one specific concept.\n"
    "- Any outlier terms should be plausibly related near-neighbors, not unrelated noise.\n"
    "- If you would need to stretch to explain how more than 2-3 terms fit the label, "
    "do not inject.\n"
    "- Assign a short snake_case primary_category.\n"
    "- Good granularity is roughly: marine_biology, sign_language, woodworking, "
    "circuit_building.\n"
    "- Too broad: biology, art, reading.\n"
    "- Too narrow: opisthokont_phylogeny.\n"
    "- subcategory: optional refinement of primary_category when a narrower, still-useful "
    "concept is clearly supported by the terms "
    "(e.g. primary_category=marine_biology, subcategory=coral_reef_ecology). "
    "Otherwise use null.\n\n"
    "split:\n"
    "- Use only when the cluster clearly contains 2-3 distinct coherent subgroups.\n"
    "- Each subgroup should have at least 3 terms.\n"
    "- You must assign the original terms into subgroup term lists.\n"
    "- If subgroup boundaries are weak or debatable, discard instead.\n\n"
    "discard:\n"
    "- Use for generic instructional language, weak thematic overlap, mixed concepts, "
    "morphological variants, noisy tokens, or overlapping senses.\n"
    "- When in doubt, discard.\n\n"
    "Examples:\n"
    "[inject] [terrarium, vivarium, reptile_habitat, snake_enclosure] "
    '-> {{"action":"inject","primary_category":"terrarium_keeping","subcategory":null,'
    '"split_into":[],"reasoning":"The terms consistently point to one specific habitat-keeping concept."}}\n'
    "[inject] [coral, reef_tank, anemone, clownfish, salinity] "
    '-> {{"action":"inject","primary_category":"marine_biology","subcategory":"coral_reef_ecology",'
    '"split_into":[],"reasoning":"The terms point to a specific marine-biology subdomain centered on reef ecosystems."}}\n'
    "[split] [watercolor, oil_paint, acrylic, welding, metalwork, soldering] "
    '-> {{"action":"split","primary_category":null,"subcategory":null,"split_into":['
    '{{"label":"painting_media","terms":["watercolor","oil_paint","acrylic"]}},'
    '{{"label":"metal_fabrication","terms":["welding","metalwork","soldering"]}}'
    '],"reasoning":"The cluster contains two clear craft/material subdomains."}}\n'
    "[discard] [learning, skill, activity, practice, work] "
    '-> {{"action":"discard","primary_category":null,"subcategory":null,"split_into":[],'
    '"reasoning":"The terms are generic instructional language rather than one specific concept."}}\n'
    "[discard] [un_funded, re_funded, pre_funded] "
    '-> {{"action":"discard","primary_category":null,"subcategory":null,"split_into":[],'
    '"reasoning":"The cluster is morphological noise rather than a coherent domain concept."}}\n\n'
    "Return a JSON object with exactly these keys:\n"
    '  "action": "inject" | "split" | "discard"\n'
    '  "primary_category": snake_case string or null\n'
    '  "subcategory": snake_case string or null\n'
    '  "split_into": [{{"label": "snake_case_label", "terms": ["term1", "term2", ...]}}, ...] '
    'if action="split", else []\n'
    '  "reasoning": one short sentence grounded in the terms\n\n'
    "Output constraints:\n"
    "- If action = inject, provide primary_category; use subcategory only if it adds useful specificity.\n"
    "- If action = split, primary_category=null and subcategory=null.\n"
    "- If action = discard, set primary_category=null, subcategory=null, split_into=[].\n"
    "- Return JSON only."
)

gate_results = []
for i, row in clusters.iterrows():
    result = gate_cluster(
        row.cluster_id, row.terms,
        client=client,
        model=MODEL_GATE,
        system_prompt=GATE_SYSTEM,
        prompt_template=GATE_PROMPT,
        retries=CFG["llm"]["max_retries"],
    )
    result["cluster_id"] = row.cluster_id
    result["terms"]      = row.terms
    gate_results.append(result)
    action = result.get("action", "?")
    label  = result.get("primary_category", "")
    if (i % 10 == 0) or action != "discard":
        print(f"  C{row.cluster_id:03d} [{action:7s}] {label or result.get('reasoning','')[:60]}")

# Schema-stable even when no clusters were processed.
GATE_SCHEMA = ["cluster_id", "terms", "action", "primary_category", "subcategory", "split_into", "reasoning"]
gate_df = (
    pd.DataFrame(gate_results)
    if gate_results
    else pd.DataFrame(columns=GATE_SCHEMA)
)
print("\nGate summary:")
if not gate_df.empty:
    print(gate_df["action"].value_counts().to_string())
else:
    print("  No clusters were gated (pass produced no results).")


  C002 [inject ] south_american_countries
  C009 [inject ] family_relatives
  C066 [discard] The terms are broadly related to nonprofit/charitable work b
  C075 [inject ] electrical_conductivity
  C079 [inject ] dodgeball
  C149 [discard] The terms are generic pronouns rather than a specific educat
  C157 [inject ] advertising
  C197 [inject ] mobile_devices
  C198 [inject ] apparel_textiles
  C294 [inject ] international_travel
  C442 [inject ] naval_warfare
  C586 [inject ] audio_story_player

Gate summary:
action
discard    20
inject     10


In [5]:
# ── Expand gate results into a flat term → category mapping ───────────────────
sem_rows = []

for _, row in gate_df.iterrows():
    if row.action == "inject" and row.primary_category:
        for term in row.terms:
            sem_rows.append({
                "term": term,
                "primary_category": row.primary_category,
                "subcategory": row.get("subcategory"),
                "source_cluster": row.cluster_id,
            })
    elif row.action == "split" and row.get("split_into"):
        for sub in row["split_into"]:
            for term in sub.get("terms", []):
                sem_rows.append({
                    "term": term,
                    "primary_category": sub["label"],
                    "subcategory": None,
                    "source_cluster": row.cluster_id,
                })

# Schema-stable even when no terms were accepted.
SEMANTIC_MAP_SCHEMA = ["term", "primary_category", "subcategory", "source_cluster"]
semantic_map = (
    pd.DataFrame(sem_rows)
    if sem_rows
    else pd.DataFrame(columns=SEMANTIC_MAP_SCHEMA)
)
semantic_map.to_csv(OUT("enrichment", "semantic_map.csv"), index=False)

print(f"semantic_map.csv: {len(semantic_map):,} terms mapped")
if not semantic_map.empty:
    print(f"Unique primary categories: {semantic_map['primary_category'].nunique()}")
    print("\nTop categories by term count:")
    print(semantic_map["primary_category"].value_counts().head(20).to_string())
else:
    print("No terms mapped — semantic_map.csv written with headers only.")

semantic_map.csv: 30 terms mapped
Unique primary categories: 10

Top categories by term count:
primary_category
south_american_countries    3
family_relatives            3
electrical_conductivity     3
dodgeball                   3
advertising                 3
mobile_devices              3
apparel_textiles            3
international_travel        3
naval_warfare               3
audio_story_player          3


---
## Pass B — Framing Taxonomy Classification

Classifies the full vocabulary against a curated framing taxonomy to identify
terms that carry rhetorical tone or contextual framing signals (urgency,
gratitude, scarcity, etc.).

**Taxonomy source:** `OUTPUTS/enrichment/framing_taxonomy.json` (curated externally).
To update the category set, edit that file directly and set
`enrichment.force_reclassify: true` in `params.yaml` before re-running.

Classification results are cached by taxonomy hash, vocabulary hash, model, and
batch size. The cache is invalidated automatically when any of these change;
`force_reclassify: true` overrides the cache regardless.

In [6]:
# ── Load curated taxonomy/enrichment file ────────────────────────────────────
# The taxonomy is developed and maintained externally. Copy the current version
# to OUTPUTS/enrichment/framing_taxonomy.json before running this cell.
# The filename remains framing_taxonomy.json for backward compatibility, but the
# file may now include namespaced framing, subject, industry, request, context,
# and sensitive_context categories.
# To update categories, edit the file and set force_reclassify: true.

taxonomy_path = OUT("enrichment", "framing_taxonomy.json")
if not taxonomy_path.exists():
    raise FileNotFoundError(
        f"framing_taxonomy.json not found at {taxonomy_path}\n"
        "Copy your curated taxonomy file to OUTPUTS/enrichment/ before running."
    )

with open(taxonomy_path) as f:
    taxonomy = json.load(f)

nmf_cats      = [c for c in taxonomy if c.get("tier") == "nmf"]
analysis_cats = [c for c in taxonomy if c.get("tier") == "analysis"]

print(f"Loaded {len(taxonomy)} categories from {taxonomy_path.name}")
print(f"  NMF-tier:      {len(nmf_cats)}")
print(f"  Analysis-tier: {len(analysis_cats)}")
print()
for cat in taxonomy:
    tier = cat.get("tier", "?")
    cov  = cat.get("target_coverage_pct", "?")
    print(f"  [{tier:8s}] {cat['category']:50s} {cov}")

Loaded 124 categories from framing_taxonomy.json
  NMF-tier:      15
  Analysis-tier: 109

  [analysis] context_alternative_school_justice_adjacent_context 2-8%
  [analysis] context_direct_attendance_absence_terms            2-8%
  [analysis] context_discipline_exclusion_absence               2-8%
  [analysis] context_mentoring_protective_support               2-8%
  [analysis] context_rural_broadband_technology_access          2-8%
  [analysis] context_rural_distance_access_barrier              2-8%
  [analysis] framing_agency_independence_responsibility         6-12%
  [analysis] framing_ai_literacy_responsible_use                2-6%
  [analysis] framing_ambitious_programmatic_ask                 6-12%
  [analysis] framing_anecdote_scene_setting                     6-12%
  [analysis] framing_applied_production_real_world_execution    5-10%
  [analysis] framing_authentic_audience_real_world              5-10%
  [nmf     ] framing_barrier_removal                            25-40%
  [a

In [7]:
# ── Build Pass B classification vocabulary ───────────────────────────────────
# Pass B should classify:
#   1. all retained unigrams, plus
#   2. taxonomy phrase anchors from examples/counter_examples that exist in
#      taxonomy_vocab_artifact.csv.
#
# It should NOT classify the full 600K+ n-gram artifact.

def normalize_anchor_term(x):
    if pd.isna(x):
        return ""
    s = str(x).strip().lower()
    s = s.replace("_", " ")
    s = re.sub(r"\s+", " ", s)
    return s.strip(" \t\r\n\"'`.,;:()[]{}")

taxonomy_anchor_terms = set()

for cat in taxonomy:
    for field in ["examples", "counter_examples"]:
        for term in cat.get(field, []):
            t = normalize_anchor_term(term)
            if t:
                taxonomy_anchor_terms.add(t)

taxonomy_phrase_anchors = {
    t for t in taxonomy_anchor_terms
    if len(t.split()) >= 2
}

artifact_terms = set(taxonomy_vocab_artifact_df["term"])
available_phrase_anchors = sorted(taxonomy_phrase_anchors & artifact_terms)
missing_phrase_anchors = sorted(taxonomy_phrase_anchors - artifact_terms)

phrase_vocab_df = (
    taxonomy_vocab_artifact_df[
        taxonomy_vocab_artifact_df["term"].isin(available_phrase_anchors)
    ]
    .set_index("term")
    [["corpus_freq", "doc_freq", "docs_pct", "term_type", "source"]]
)

# Pass B vocabulary = unigrams + taxonomy-relevant phrase anchors.
pass_b_vocab_df = pd.concat(
    [unigram_vocab_df, phrase_vocab_df],
    axis=0
)

pass_b_vocab_df = pass_b_vocab_df[~pass_b_vocab_df.index.duplicated(keep="first")]
pass_b_vocab_df = pass_b_vocab_df.sort_values(["term_type", "doc_freq"], ascending=[True, False])

# From this point through Pass B and Pass C, vocab_df represents the taxonomy
# classification/injection vocabulary.
vocab_df = pass_b_vocab_df.copy()

print(f"Pass B vocabulary terms             : {len(vocab_df):,}")
print(f"  Unigrams                          : {(vocab_df['term_type'] == 'unigram').sum():,}")
print(f"  Taxonomy phrase anchors available : {len(available_phrase_anchors):,}")
print(f"  Taxonomy phrase anchors missing   : {len(missing_phrase_anchors):,}")

if available_phrase_anchors:
    print("\nExample available phrase anchors:")
    print(available_phrase_anchors[:30])

if missing_phrase_anchors:
    print("\nExample missing phrase anchors:")
    print(missing_phrase_anchors[:30])

Pass B vocabulary terms             : 8,406
  Unigrams                          : 7,553
  Taxonomy phrase anchors available : 853
  Taxonomy phrase anchors missing   : 0

Example available phrase anchors:
['abstract concrete', 'academic integrity', 'academic intervention', 'academic skill', 'academic social', 'access barrier', 'across room', 'activity explore', 'activity strengthen', 'adaptive software', 'adaptive special', 'adaptive switch', 'adult mentor', 'agriculture sustainability', 'air purifier', 'air quality', 'alternative education', 'alternative pathway', 'alternative program', 'alternative setting', 'animal science', 'apartment complex', 'appreciate thank', 'area development', 'area read', 'area rural', 'area stem', 'art project', 'art read', 'art supply']


In [8]:
BATCH_SIZE       = CFG["enrichment"]["classify_batch_size"]
FORCE_RECLASSIFY = CFG["enrichment"]["force_reclassify"]
framing_map_path  = OUT("enrichment", "framing_map.csv")
framing_meta_path = OUT("enrichment", "framing_map_meta.json")

# ── Cache keys for this run ───────────────────────────────────────────────────
with open(OUT("enrichment", "framing_taxonomy.json")) as _f:
    current_taxonomy_hash = hashlib.md5(_f.read().encode()).hexdigest()

# vocab_md5 captures the current vocabulary so a corpus change (e.g. re-running
# NB01 with different thresholds) invalidates the cache even when the taxonomy
# file is unchanged.
vocab_md5 = hashlib.md5(",".join(sorted(vocab_df.index.tolist())).encode()).hexdigest()

def _meta_hash_matches():
    """Return True if the cached framing_map matches all current inputs.

    Validates taxonomy version, vocabulary, model, and batch size so that
    any change to the corpus or classification config invalidates the cache.
    """
    if not framing_meta_path.exists():
        return False
    with open(framing_meta_path) as _mf:
        meta = json.load(_mf)
    return (
        meta.get("taxonomy_md5")           == current_taxonomy_hash
        and meta.get("vocab_md5")          == vocab_md5
        and meta.get("model")              == MODEL_CLASSIFY
        and meta.get("classify_batch_size") == BATCH_SIZE
    )

# ── Build taxonomy reference string and valid category set for prompts ────────
def _fmt_cat(cat):
    return "\n".join([
        f"  [{cat.get('tier','?')}] {cat['category']} (target: {cat.get('target_coverage_pct','?')})",
        f"    Description: {cat['description']}",
        f"    Examples: {', '.join(cat.get('examples', [])[:8])}",
        f"    Counter-examples (must return null): {', '.join(cat.get('counter_examples', [])[:6])}",
    ])

taxonomy_ref     = "\n\n".join(_fmt_cat(cat) for cat in taxonomy)
valid_categories = {cat["category"] for cat in taxonomy}

CLASSIFY_SYSTEM = (
    "You classify vocabulary terms from DonorsChoose K-12 teacher project essays into a "
    "closed set of taxonomy/enrichment categories.\n\n"
    "Your output drives injection of namespace-aware synthetic tokens into project token lists. "
    "Those injected tokens become downstream topic-modeling and analysis signals. "
    "Categories may represent rhetorical framing, subject/domain themes, workforce industries, "
    "request/material logic, contextual circumstances, or sensitive-context signals.\n\n"
    "Category namespace conventions:\n"
    "- framing_* = rhetorical framing, tone, persuasion logic, or mechanism-of-need signal\n"
    "- subject_* = subject/domain/content signal\n"
    "- industry_* = workforce-development industry or skill-domain facet\n"
    "- request_* = material/request topology signal\n"
    "- context_* = non-sensitive contextual condition or school/community circumstance\n"
    "- sensitive_context_* = sensitive identity, justice, disability, or similar direct-context signal\n\n"
    "Core rule:\n"
    "- Classify a term only when it clearly fits one supplied category by the category description.\n"
    "- Use subject/domain categories for WHAT the project is about only when the supplied category "
    "explicitly asks for that subject/domain signal.\n"
    "- Use framing/context categories for HOW the teacher frames the request, need, circumstance, "
    "or mechanism.\n"
    "- Use sensitive_context categories only for direct, explicit signals. Never infer sensitive "
    "context from vague proxy terms.\n\n"
    "Most terms should be null.\n"
    "Classify a term only when its category meaning is evident without needing sentence context — "
    "you could label it correctly knowing only the term.\n\n"
    "Use only the supplied taxonomy categories. Never invent a category. Prefer null "
    "over a weak guess.\n\n"
    "Return ONLY a valid JSON object mapping every input term to either one exact "
    "taxonomy category name or null. No markdown, no code fences, no commentary."
)

CLASSIFY_PROMPT = (
    "Classify each term into exactly one taxonomy/enrichment category or null.\n\n"
    "Expected sparsity:\n"
    "- Only a minority of terms should be classified.\n"
    "- Err heavily toward null when the term is generic, ambiguous, too broad, or dependent "
    "on sentence context.\n\n"
    "Classify a term only when ALL of the following are true:\n"
    "1. The term clearly fits one supplied category description.\n"
    "2. The category meaning is evident without needing sentence context.\n"
    "3. The term is specific enough to serve as a reliable source term for an injected signal.\n\n"
    "Use null when the term is:\n"
    "- generic school or classroom language\n"
    "- too broad to distinguish projects\n"
    "- merely descriptive without category-specific meaning\n"
    "- ambiguous across multiple categories\n"
    "- dependent on sentence context to carry the category meaning\n"
    "- a vague proxy for a sensitive-context category\n\n"
    "Namespace guidance:\n"
    "- framing_* categories capture rhetorical framing, tone, mechanism, or persuasion logic.\n"
    "- subject_* categories capture domain/content/topic signals.\n"
    "- industry_* categories capture workforce-development industry or skill facets.\n"
    "- request_* categories capture material or request-topology signals.\n"
    "- context_* categories capture contextual school, community, attendance, safety, or access conditions.\n"
    "- sensitive_context_* categories require direct, explicit signals; do not infer them from proxies.\n\n"
    "Taxonomy categories:\n"
    "{taxonomy}\n\n"
    "Terms: {terms}\n\n"
    "Return exactly one JSON object of this form:\n"
    '{{"term1": "exact_taxonomy_category_name", "term2": null}}\n'
    "Include every input term as a key. The example above shows only two keys for brevity.\n\n"
    "Requirements:\n"
    "- Every input term must appear exactly once as a key.\n"
    "- Use only exact category names from the taxonomy.\n"
    "- Do not explain your answers.\n"
    "- Return JSON only."
)

# ── Classify or load from cache ───────────────────────────────────────────────
if not FORCE_RECLASSIFY and framing_map_path.exists() and _meta_hash_matches():
    framing_map = pd.read_csv(framing_map_path)
    print(f"Cache hit — reusing framing_map.csv ({len(framing_map):,} terms classified)")
else:
    if framing_map_path.exists() and not _meta_hash_matches():
        print("Cache invalid — re-classifying (taxonomy, vocab, model, or batch size changed).")

    all_terms   = vocab_df.index.tolist()
    batches     = [all_terms[i:i + BATCH_SIZE] for i in range(0, len(all_terms), BATCH_SIZE)]
    framing_raw = {}

    print(f"Classifying {len(all_terms):,} terms in {len(batches)} batches...")
    for i, batch in enumerate(batches):
        result = classify_batch(
            batch,
            client=client,
            model=MODEL_CLASSIFY,
            system_prompt=CLASSIFY_SYSTEM,
            prompt_template=CLASSIFY_PROMPT,
            taxonomy_ref=taxonomy_ref,
            valid_categories=valid_categories,
            retries=CFG["llm"]["max_retries"],
        )
        framing_raw.update(result)
        if (i + 1) % 10 == 0:
            classified = sum(1 for v in framing_raw.values() if v is not None)
            print(f"  Batch {i+1}/{len(batches)} — {classified:,} terms classified so far")

    framing_map = pd.DataFrame([
        {"term": term, "framing_category": cat}
        for term, cat in framing_raw.items()
        if cat is not None
    ])
    framing_map.to_csv(framing_map_path, index=False)

# ── Persist provenance metadata ───────────────────────────────────────────────
meta = {
    "taxonomy_md5":        current_taxonomy_hash,
    "vocab_md5":           vocab_md5,
    "model":               MODEL_CLASSIFY,
    "classify_batch_size": BATCH_SIZE,
    "classified_at":       datetime.now().isoformat(),
    "n_terms_classified":  int(len(framing_map)),
    "n_categories":        int(framing_map["framing_category"].nunique()) if len(framing_map) else 0,
    "coverage_pct":        round(len(framing_map) / len(vocab_df) * 100, 2) if len(vocab_df) else 0.0,
}
with open(framing_meta_path, "w", encoding="utf-8") as f:
    json.dump(meta, f, indent=2)

print(f"\nframing_map.csv : {len(framing_map):,} terms classified")
print(f"Vocab coverage  : {len(framing_map)/len(vocab_df):.1%}")
if not framing_map.empty:
    print("\nCategory distribution:")
    print(framing_map["framing_category"].value_counts().to_string())
else:
    print("No terms classified — framing_map.csv is empty.")

# ── Coverage deviation warnings ───────────────────────────────────────────────
# Warn when a category's realized coverage deviates from its taxonomy target by
# more than coverage_deviation_warn_threshold percentage points.
warn_threshold    = CFG["enrichment"]["coverage_deviation_warn_threshold"]
actual_pct_by_cat = (
    framing_map["framing_category"].value_counts(normalize=True).mul(100).to_dict()
    if not framing_map.empty
    else {}
)

def _target_midpoint(v):
    """Parse a target coverage string like '2-4%' to its numeric midpoint."""
    if v is None:
        return None
    s = str(v).strip().replace("%", "")
    if "-" in s:
        lo, hi = s.split("-", 1)
        try:
            return (float(lo) + float(hi)) / 2
        except ValueError:
            return None
    try:
        return float(s)
    except ValueError:
        return None

for cat in taxonomy:
    target = _target_midpoint(cat.get("target_coverage_pct"))
    if target is None:
        continue
    actual = actual_pct_by_cat.get(cat["category"], 0.0)
    if abs(actual - target) > warn_threshold:
        append_warning(
            WARNINGS_PATH, "02_semantic_enrichment", "COVERAGE_DEVIATION",
            f"Coverage deviation for {cat['category']}",
            context={"category": cat["category"], "actual_pct": actual, "target_pct": target},
        )
        print(
            f"WARNING: {cat['category']}: "
            f"actual={actual:.1f}% vs target\u2248{target:.1f}% "
            f"(threshold \u00b1{warn_threshold}pp)"
        )


Cache invalid — re-classifying (taxonomy, vocab, model, or batch size changed).
Classifying 8,406 terms in 57 batches...
  Batch 10/57 — 882 terms classified so far
  Batch 20/57 — 1,293 terms classified so far
  Batch 30/57 — 1,752 terms classified so far
  Batch 40/57 — 2,112 terms classified so far
  Batch 50/57 — 2,393 terms classified so far

framing_map.csv : 2,587 terms classified
Vocab coverage  : 30.8%

Category distribution:
framing_category
framing_place_community_specific_context               108
framing_joy_possibility_appeal                          67
framing_durable_infrastructure                          51
framing_household_basic_needs_spillover                 50
framing_expressive_creative_production                  49
framing_capacity_first_dignity                          48
framing_sensory_detail                                  48
industry_media_creative_industries                      47
framing_standards_research_justification                43
framing_regul

---
## Pass C — Token Injection

Builds an injection lookup from Pass A and Pass B results, then appends
enrichment tokens to each project's token list. Injection is strictly
additive — original tokens are never modified or removed.

Token naming convention:
- `__cat_marine_biology__` — subject category (Pass A inject)
- `__sub_ocean_ecosystems__` — subject subcategory (Pass A split)
- `__framing_urgency__` — framing signal (Pass B)

The double-underscore prefix ensures injected tokens are visually distinct
in all downstream outputs and can be filtered or analyzed independently.

In [9]:
# ── Load injection config ─────────────────────────────────────────────────────
# Ceiling exceptions are high-frequency terms that bypass max_doc_freq_for_injection
# because they are definitionally load-bearing for their framing category.
# Source: params.yaml → enrichment.ceiling_exceptions_path
ceiling_path = ROOT / CFG["enrichment"]["ceiling_exceptions_path"]
with open(ceiling_path) as f:
    ceiling_exceptions = yaml.safe_load(f)
print(f"Ceiling exceptions : {len(ceiling_exceptions)} terms from {ceiling_path.name}")

MIN_INJECT_PROJECTS     = CFG["enrichment"]["min_inject_projects"]
MAX_DOC_FREQ_FOR_INJECT = CFG["enrichment"]["max_doc_freq_for_injection"]

# ── Reload maps from disk ─────────────────────────────────────────────────────
semantic_map = pd.read_csv(OUT("enrichment", "semantic_map.csv"))
framing_map  = pd.read_csv(OUT("enrichment", "framing_map.csv"))

# Reload valid_categories from the on-disk taxonomy so Pass C is self-contained
# and does not depend on the Pass B in-memory variable. This makes it safe to
# re-run Pass C independently without re-running Pass B first.
with open(OUT("enrichment", "framing_taxonomy.json")) as _f:
    _tax = json.load(_f)
valid_categories = {cat["category"] for cat in _tax}

# Validate that all framing categories are present in the loaded taxonomy.
invalid_cats = set(framing_map["framing_category"]) - valid_categories
if invalid_cats:
    raise ValueError(
        f"framing_map.csv contains categories not in framing_taxonomy.json: "
        f"{sorted(invalid_cats)}"
    )

# ── Build subject injection lookup (Pass A → __cat_* / __sub_*) ───────────────
inject_lookup = {}

for _, row in semantic_map.iterrows():
    if row.term not in vocab_df.index:
        continue
    if vocab_df.loc[row.term, "doc_freq"] > MAX_DOC_FREQ_FOR_INJECT:
        continue
    tokens = [f"__cat_{row.primary_category}__"]
    if pd.notna(row.get("subcategory")) and row.subcategory:
        tokens.append(f"__sub_{row.subcategory}__")
    inject_lookup.setdefault(row.term, []).extend(tokens)

# ── Build taxonomy injection lookup (Pass B → namespace-aware tokens) ─────────
# Category names may be prefixed with a namespace:
#   framing_*
#   subject_*
#   industry_*
#   request_*
#   context_*
#   sensitive_context_*
#
# These become injected tokens like:
#   framing_barrier_removal                  → __framing_barrier_removal__
#   subject_ai_literacy_foundations          → __subject_ai_literacy_foundations__
#   industry_health_careers                  → __industry_health_careers__
#   request_basic_classroom_supplies         → __request_basic_classroom_supplies__
#   context_direct_attendance_absence_terms  → __context_direct_attendance_absence_terms__
#   sensitive_context_lgbtq_affirming_representation
#                                            → __sensitive_context_lgbtq_affirming_representation__
#
# Backward compatibility:
#   Unprefixed legacy categories are treated as framing categories.

CATEGORY_NAMESPACES = [
    "sensitive_context",
    "framing",
    "subject",
    "industry",
    "request",
    "context",
]

def injected_token_for_category(category: str) -> str:
    """Convert a taxonomy category name into a namespace-aware injected token."""
    category = str(category).strip()

    for ns in CATEGORY_NAMESPACES:
        prefix = f"{ns}_"
        if category.startswith(prefix):
            # Keep the full category name in the token so names remain globally unique.
            return f"__{category}__"

    # Legacy fallback for old framing_taxonomy.json categories like barrier_removal.
    return f"__framing_{category}__"


for _, row in framing_map.iterrows():
    if row.term not in vocab_df.index:
        continue
    if vocab_df.loc[row.term, "doc_freq"] > MAX_DOC_FREQ_FOR_INJECT:
        # High-frequency terms only bypass the ceiling when they are a listed
        # load-bearing exception for their specific category.
        if ceiling_exceptions.get(row.term) != row.framing_category:
            continue

    inject_lookup.setdefault(row.term, []).append(
        injected_token_for_category(row.framing_category)
    )

# ── Deduplicate tokens per term and apply min_inject_projects filter ──────────
# Filter is applied at the source-term level: a source term whose document
# frequency is below min_inject_projects is excluded from inject_lookup because
# it appears too rarely to contribute meaningful signal, even if it shares an
# injected token with higher-frequency terms.
# (See params.yaml → enrichment.min_inject_projects)
inject_lookup = {
    term: list(dict.fromkeys(tokens))
    for term, tokens in inject_lookup.items()
    if vocab_df.loc[term, "doc_freq"] >= MIN_INJECT_PROJECTS
}

unique_injected = {tok for toks in inject_lookup.values() for tok in toks}
print(f"Source terms with injections : {len(inject_lookup):,}")
print(f"Unique injected tokens        : {len(unique_injected):,}")

Ceiling exceptions : 31 terms from ceiling_exceptions.yaml
Source terms with injections : 2,257
Unique injected tokens        : 136


In [10]:
# ── Estimate injected token coverage before writing ───────────────────────────
# Counts how many projects will receive each injected token, and flags any
# that will fall below sql.min_doc_count after NB03's TF-IDF step.

min_doc_count = CFG["sql"]["min_doc_count"]

def normalize_project_token(x):
    if pd.isna(x):
        return ""
    s = str(x).strip().lower()
    s = s.replace("_", " ")
    s = re.sub(r"\s+", " ", s)
    return s.strip(" \t\r\n\"'`.,;:()[]{}")

def source_terms_for_injection(tokens, inject_lookup):
    """
    Return unigram source terms plus phrase source terms that are present in
    inject_lookup.

    This allows a project token sequence like ["generative", "ai"] to match
    an inject_lookup source term like "generative ai".
    """
    toks = [normalize_project_token(t) for t in list(tokens) if normalize_project_token(t)]
    source_terms = set(toks)

    phrase_terms = [t for t in inject_lookup.keys() if " " in str(t)]
    if phrase_terms:
        max_n = max(len(str(t).split()) for t in phrase_terms)
        max_n = min(max_n, 3)

        for n in range(2, max_n + 1):
            if len(toks) < n:
                continue
            for i in range(0, len(toks) - n + 1):
                phrase = " ".join(toks[i:i+n])
                if phrase in inject_lookup:
                    source_terms.add(phrase)

    return source_terms

injected_token_counts: dict[str, int] = {}
for token_list in df["tokens"]:
    seen: set[str] = set()
    for source_term in source_terms_for_injection(token_list, inject_lookup):
        if source_term in inject_lookup:
            seen.update(inject_lookup[source_term])
    for inj in seen:
        injected_token_counts[inj] = injected_token_counts.get(inj, 0) + 1

# Schema-stable even when inject_lookup is empty.
_inj_rows = [
    {"injected_token": tok, "project_count": cnt}
    for tok, cnt in injected_token_counts.items()
]
inj_freq_df = (
    pd.DataFrame(_inj_rows).sort_values("project_count", ascending=False)
    if _inj_rows
    else pd.DataFrame(columns=["injected_token", "project_count"])
)

will_survive = inj_freq_df[inj_freq_df["project_count"] >= min_doc_count]
will_drop    = inj_freq_df[inj_freq_df["project_count"] <  min_doc_count]

print(f"Injected tokens surviving min_doc_count={min_doc_count}: {len(will_survive)}")
if not will_drop.empty:
    print(f"WARNING: {len(will_drop)} injected tokens below threshold (will be filtered in NB03):")
    print(f"  {will_drop['injected_token'].tolist()}")
if not inj_freq_df.empty:
    print("\nTop injected tokens by project coverage:")
    print(will_survive.head(200).to_string(index=False))
else:
    print("No injections to preview (inject_lookup is empty).")

Injected tokens surviving min_doc_count=15: 136

Top injected tokens by project coverage:
                                         injected_token  project_count
           __framing_place_community_specific_context__         258958
                     __framing_joy_possibility_appeal__         190925
                     __framing_durable_infrastructure__         190226
                     __framing_capacity_first_dignity__         173173
                             __framing_loss_avoidance__         143812
              __framing_milestone_culmination_framing__         133371
                       __framing_numeracy_math_skills__         130211
             __framing_belonging_identity_affirmation__         129530
             __framing_expressive_creative_production__         126720
                             __framing_sensory_detail__         125001
        __framing_foundational_communication_literacy__         121028
            __framing_household_basic_needs_spillover__   

In [12]:
# ── Inject, derive project-level taxonomy tags, and write enriched parquet ────
# This replaces the simple inject_tokens() call so we can also preserve
# project-level taxonomy tags for filtering/groupby in downstream notebooks.
#
# Interpretation:
#   taxonomy_tags = list of taxonomy categories whose injected token appears
#                   in the project after enrichment.
#   taxonomy_tags_json = JSON string version of taxonomy_tags.
#   taxonomy_tag_source_terms_json = audit/debug field showing which original
#                                    source terms caused each taxonomy tag.
#   tag_[category] = project-level boolean field for easy filtering/groupby.
#
# These are "likely related" tags, not deterministic labels.

from collections import defaultdict
import json
import re

TAXONOMY_INJECTED_TOKEN_RE = re.compile(
    r"^__(framing|subject|industry|request|context|sensitive_context)_(.+)__$"
)

def taxonomy_category_from_injected_token(token: str):
    """
    Convert an injected taxonomy token to a project-level tag/category name.

    Examples:
      __framing_barrier_removal__                  → framing_barrier_removal
      __subject_ai_literacy_foundations__          → subject_ai_literacy_foundations
      __industry_health_careers__                  → industry_health_careers
      __request_basic_classroom_supplies__         → request_basic_classroom_supplies
      __context_direct_attendance_absence_terms__  → context_direct_attendance_absence_terms
      __sensitive_context_lgbtq_affirming_representation__
                                                    → sensitive_context_lgbtq_affirming_representation

    Returns None for legacy Pass A subject tokens:
      __cat_*__
      __sub_*__
    """
    token = str(token)
    m = TAXONOMY_INJECTED_TOKEN_RE.match(token)
    if not m:
        return None

    namespace = m.group(1)
    remainder = m.group(2)
    return f"{namespace}_{remainder}"


def inject_tokens_with_taxonomy_support(tokens, inject_lookup):
    """
    Add injected tokens to a project's token list while preserving taxonomy
    tag source-term support.

    Uses both unigram source terms and taxonomy phrase source terms. Deduplicates
    injected tokens per project so each injected category token appears at most
    once in the enriched token list.
    """
    original_tokens = list(tokens)

    injected_tokens = set()
    taxonomy_support = defaultdict(set)

    for source_term in source_terms_for_injection(original_tokens, inject_lookup):
        for injected_token in inject_lookup.get(source_term, []):
            injected_tokens.add(injected_token)

            category = taxonomy_category_from_injected_token(injected_token)
            if category is not None:
                taxonomy_support[category].add(source_term)

    enriched_tokens = original_tokens + sorted(injected_tokens)
    taxonomy_support = {
        category: sorted(source_terms)
        for category, source_terms in taxonomy_support.items()
    }

    return enriched_tokens, taxonomy_support


df_enriched = df.copy()

_enrichment_results = df_enriched["tokens"].apply(
    lambda ts: inject_tokens_with_taxonomy_support(list(ts), inject_lookup)
)

df_enriched["tokens"] = _enrichment_results.apply(lambda x: x[0])

# Keep source-term support as JSON, not a raw dict column, to avoid parquet
# compatibility issues with variable dictionary keys.
df_enriched["taxonomy_tag_source_terms_json"] = _enrichment_results.apply(
    lambda x: json.dumps(x[1], sort_keys=True)
)

df_enriched["taxonomy_tags"] = _enrichment_results.apply(
    lambda x: sorted(x[1].keys())
)
df_enriched["taxonomy_tags_json"] = df_enriched["taxonomy_tags"].apply(json.dumps)

# Create one boolean field per taxonomy category so 04 can filter/groupby
# without parsing token lists.
all_taxonomy_tags = sorted({
    tag
    for tags in df_enriched["taxonomy_tags"]
    for tag in tags
})

for tag in all_taxonomy_tags:
    df_enriched[f"tag_{tag}"] = df_enriched["taxonomy_tags"].apply(
        lambda tags, t=tag: t in tags
    )

orig_len          = df["tokens"].apply(len)
new_len           = df_enriched["tokens"].apply(len)
enriched_projects = (new_len > orig_len).sum()
tagged_projects   = df_enriched["taxonomy_tags"].apply(bool).sum()

print(f"Projects with injected tokens       : {enriched_projects:,} ({enriched_projects/len(df):.1%})")
print(f"Projects with taxonomy tags         : {tagged_projects:,} ({tagged_projects/len(df):.1%})")
print(f"Unique taxonomy tag fields added    : {len(all_taxonomy_tags):,}")
print(f"Mean tokens before injection        : {orig_len.mean():.1f}")
print(f"Mean tokens after injection         : {new_len.mean():.1f}")
print(f"Max injected per project            : {(new_len - orig_len).max()}")

if all_taxonomy_tags:
    print("\nExample taxonomy tag fields:")
    for tag in all_taxonomy_tags[:20]:
        print(f"  tag_{tag}")

out_path = OUT("prepared", "06_enriched.parquet")
df_enriched.to_parquet(out_path, index=False)
print(f"\nSaved → {out_path}")
print("\nTo use the enriched corpus in 03_insights_generation, load:")
print('  df = pd.read_parquet(ROOT / "OUTPUTS/prepared/06_enriched.parquet")')

# ── Stage manifest ────────────────────────────────────────────────────────────
stage_manifest_path = OUT("enrichment/metadata", "stage_manifest_02_semantic_enrichment.json")
finalize_stage_manifest(
    STAGE_MANIFEST,
    output_path=stage_manifest_path,
    status="success",
    input_artifacts=[
        artifact_meta(ROOT / "OUTPUTS/prepared/05_filtered.parquet",  "filtered_parquet"),
        artifact_meta(ROOT / "OUTPUTS/vocab/unigram_docfreq.csv",     "unigram_docfreq_csv"),
        artifact_meta(OUT("enrichment", "framing_taxonomy.json"),      "framing_taxonomy_json"),
    ],
    output_artifacts=[
        artifact_meta(OUT("enrichment", "semantic_map.csv"),            "semantic_map_csv"),
        artifact_meta(OUT("enrichment", "framing_map.csv"),             "framing_map_csv"),
        artifact_meta(OUT("enrichment", "framing_map_meta.json"),       "framing_map_meta_json"),
        artifact_meta(OUT("vocab",      "taxonomy_vocab_artifact.csv"), "taxonomy_vocab_artifact_csv"),
        artifact_meta(OUT("prepared",   "06_enriched.parquet"),         "enriched_parquet"),
    ],
    row_counts={
        "input_projects":                    int(len(df)),
        "enriched_projects":                 int(enriched_projects),
        "taxonomy_tagged_projects":          int(tagged_projects),
        "taxonomy_tag_fields":               int(len(all_taxonomy_tags)),
        "semantic_terms_mapped":             int(len(semantic_map)),
        "framing_terms_mapped":              int(len(framing_map)),
        "inject_source_terms":               int(len(inject_lookup)),
        "taxonomy_vocab_artifact_terms":     int(len(taxonomy_vocab_artifact_df)),
        "pass_b_vocab_terms":                int(len(vocab_df)),
        "pass_b_phrase_anchor_terms":        int(len([t for t in vocab_df.index if " " in str(t)])),
    },
    key_params={
        "rare_doc_freq_ceiling":      CFG["enrichment"]["rare_doc_freq_ceiling"],
        "agg_distance_threshold":     CFG["enrichment"]["agg_distance_threshold"],
        "min_cluster_size":           CFG["enrichment"]["min_cluster_size"],
        "embedding_model":            CFG["enrichment"]["embedding_model"],
        "classify_batch_size":        CFG["enrichment"]["classify_batch_size"],
        "max_doc_freq_for_injection": CFG["enrichment"]["max_doc_freq_for_injection"],
        "min_inject_projects":        CFG["enrichment"]["min_inject_projects"],
    },
    warnings_path=WARNINGS_PATH,
)
print(f"Stage manifest → {stage_manifest_path}")

/var/folders/j3/jwjf6cwj7czdz1klxbhhjst80000gp/T/ipykernel_658/2039886019.py:109: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_enriched[f"tag_{tag}"] = df_enriched["taxonomy_tags"].apply(
/var/folders/j3/jwjf6cwj7czdz1klxbhhjst80000gp/T/ipykernel_658/2039886019.py:109: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_enriched[f"tag_{tag}"] = df_enriched["taxonomy_tags"].apply(
/var/folders/j3/jwjf6cwj7czdz1klxbhhjst80000gp/T/ipykernel_658/2039886019.py:109: PerformanceWarning: DataFrame is highly fragmented.  This is usually

Projects with injected tokens       : 2,067,410 (92.1%)
Projects with taxonomy tags         : 2,067,163 (92.1%)
Unique taxonomy tag fields added    : 124
Mean tokens before injection        : 58.4
Mean tokens after injection         : 61.6
Max injected per project            : 20

Example taxonomy tag fields:
  tag_context_alternative_school_justice_adjacent_context
  tag_context_direct_attendance_absence_terms
  tag_context_discipline_exclusion_absence
  tag_context_mentoring_protective_support
  tag_context_rural_broadband_technology_access
  tag_context_rural_distance_access_barrier
  tag_framing_agency_independence_responsibility
  tag_framing_ai_literacy_responsible_use
  tag_framing_ambitious_programmatic_ask
  tag_framing_anecdote_scene_setting
  tag_framing_applied_production_real_world_execution
  tag_framing_authentic_audience_real_world
  tag_framing_barrier_removal
  tag_framing_basic_sufficiency
  tag_framing_belonging_identity_affirmation
  tag_framing_capacity_first_dign